# 네이버 뉴스 API를 활용한 데이터 수집

이 노트북은 네이버 뉴스 API를 활용하여 특정 키워드에 대한 뉴스 기사를 검색하고, 검색 결과를 엑셀 파일로 저장하는 방법을 보여줍니다.

## 학습 목표
1. API를 활용한 데이터 수집 방법 이해
2. JSON 데이터 처리 및 파싱 방법 학습
3. HTML 태그 제거 및 텍스트 추출 과정 학습
4. 엑셀 파일로 데이터 저장 방법 습득

## 1. 필요한 라이브러리 임포트

데이터 수집 및 처리에 필요한 라이브러리들을 임포트합니다.

In [1]:
# 필요한 라이브러리 임포트
import requests  # HTTP 요청을 보내기 위한 라이브러리
import json  # JSON 데이터 처리를 위한 라이브러리
from bs4 import BeautifulSoup  # HTML 파싱을 위한 라이브러리
import pandas as pd  # 데이터 처리를 위한 라이브러리
import re  # 정규표현식을 사용하기 위한 라이브러리
import openpyxl  # 엑셀 파일 처리를 위한 라이브러리

## 2. API 인증 정보 설정

네이버 개발자 센터(https://developers.naver.com)에서 발급받은 API 인증 정보를 설정합니다.

**주의**: 실제 수업에서는 아래 API 키를 학생들에게 제공하거나, 각자 개발자 센터에서 발급받도록 안내해야 합니다.

In [2]:
# 네이버 개발자 센터에서 발급받은 API 인증 정보
client_id = "yR8aViAqV5NyEjZpfo74"  # 네이버 API 클라이언트 ID
client_secret = "SQ_PTt2epk"  # 네이버 API 클라이언트 시크릿

## 3. 검색어 입력 및 API 요청 설정

사용자로부터 검색어를 입력받고, API 요청에 필요한 URL과 헤더를 설정합니다.

In [3]:
# 사용자로부터 검색어 입력 받기
query = input("검색할 키워드를 입력하세요: ")  # 검색할 키워드 입력
display = 10  # 한 번에 가져올 검색 결과 개수 (최대 100개까지 설정 가능)

# 네이버 뉴스 검색 API URL 생성
url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={display}"

# API 요청 헤더 설정
headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret,
}

## 4. API 요청 및 응답 처리

설정한 URL과 헤더를 사용하여 API 요청을 보내고, 응답을 처리합니다.

In [4]:
# API 요청 보내기
response = requests.get(url, headers=headers)

# 응답 상태 코드 확인
print(f"응답 상태 코드: {response.status_code}")

# 응답이 성공적인 경우 (상태 코드 200)
if response.status_code == 200:
    # JSON 응답 데이터 파싱
    news_data = response.json()
    
    # 검색 결과 항목 수 출력
    if 'items' in news_data:
        print(f"총 {len(news_data['items'])}개의 뉴스 기사를 찾았습니다.")
    else:
        print("검색 결과가 없습니다.")
else:
    print(f"API 요청 실패: {response.status_code}")

응답 상태 코드: 200
총 10개의 뉴스 기사를 찾았습니다.


## 5. 검색 결과 데이터 확인

API 응답으로 받은 JSON 데이터의 구조를 확인합니다.

In [5]:
# API 응답 데이터 구조 확인 (첫 번째 항목만)
if response.status_code == 200 and 'items' in news_data and len(news_data['items']) > 0:
    print("첫 번째 뉴스 항목 데이터:")
    print(json.dumps(news_data['items'][0], indent=2, ensure_ascii=False))
else:
    print("데이터를 표시할 수 없습니다.")

첫 번째 뉴스 항목 데이터:
{
  "title": "[6월 21일] 동방정사의 오늘의 운세",
  "originallink": "https://www.idaegu.co.kr/news/articleView.html?idxno=513522",
  "link": "https://www.idaegu.co.kr/news/articleView.html?idxno=513522",
  "description": "61년생 <b>여행</b>을 계획중이면 신중하게 생각토록 하자. 행한의 흐름이 좋지 못하니 밖으로의 출행이 그리 좋은... 86년생 이동운 <b>여행</b>운이 들어오는구나. 먼길 떠날 수 있는데 마음 상하는 일 있을까 우려되는구나, 기대가... ",
  "pubDate": "Fri, 20 Jun 2025 21:54:00 +0900"
}


## 6. 검색 결과 처리 및 엑셀 파일 저장

검색 결과에서 제목과 본문을 추출하고, HTML 태그를 제거한 후 엑셀 파일로 저장합니다.

In [6]:
if response.status_code == 200:
    # Openpyxl을 사용하여 새 워크북 생성
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "네이버 뉴스 검색 결과"
    
    # 열 제목 추가
    ws.append(["제목", "본문"])
    
    if 'items' in news_data:
        # 검색 결과가 있는 경우 처리 시작
        for item in news_data['items']:
            # HTML 강조 태그(<b></b>)를 제거하여 제목과 본문 텍스트 추출
            title = item['title'].replace("<b>", "").replace("</b>", "")
            content = item['description'].replace("<b>", "").replace("</b>", "")
            
            # BeautifulSoup을 사용하여 남아있는 HTML 태그를 제거하고 순수 텍스트 추출
            clean_title = BeautifulSoup(title, "html.parser").text
            clean_content = BeautifulSoup(content, "html.parser").text
            
            # 추출한 제목과 본문을 콘솔에 출력
            print(f"제목: {clean_title}")
            print(f"본문: {clean_content}")
            print("-" * 50)
            
            # 추출한 제목과 본문을 엑셀 워크시트에 추가
            ws.append([clean_title, clean_content])
        
        # 엑셀 파일 저장
        file_name = f"{query}_news.xlsx"
        wb.save(file_name)
        print(f"'{file_name}' 파일에 저장되었습니다.")
    else:
        # 검색 결과가 없는 경우 메시지 출력
        print("검색 결과가 없습니다.")
else:
    # API 요청이 실패한 경우 오류 코드와 함께 메시지 출력
    print("API 요청 실패:", response.status_code)

제목: [6월 21일] 동방정사의 오늘의 운세
본문: 61년생 여행을 계획중이면 신중하게 생각토록 하자. 행한의 흐름이 좋지 못하니 밖으로의 출행이 그리 좋은... 86년생 이동운 여행운이 들어오는구나. 먼길 떠날 수 있는데 마음 상하는 일 있을까 우려되는구나, 기대가... 
--------------------------------------------------
제목: 김연아, 시간이 멈춘 듯한 청량 비주얼... 은퇴 후에도 영원한 ‘퀸’
본문: 고우림은 지난 5월 19일 육군 군악대 복무를 마치고 만기 전역했으며 전역 직후 김연아와 프랑스 파리로 9박 10일간 여행을 다녀왔다고 밝히며 변함없는 애정을 과시하기도 했다.
--------------------------------------------------
제목: CJ그룹, 베트남서 'CJ K FESTA' 개최
본문: 15만VND(한화 약 8000원) 이상 구매 후 응모한 고객 중 추첨을 통해 한국 여행권, 전기 오토바이, 갤럭시 스마트폰, CGV 1년 관람권 등 총 28명의 고객에게 경품을 제공하는 빅뱅 프로모션 등 풍성한 이벤트도... 
--------------------------------------------------
제목: “택시비 없으면 몸으로…” 한국 택시기사, 태국 여행객 성희롱 ‘충격...
본문: 한국인 택시기사가 태국 여성 관광객에게 성희롱 발언을 한 것이 알려지면서 논란이 일고 있다. 19일 한 태국인 여성은 자신의 틱톡에 “한국에서 택시를 탈 때 조심하라”며 친구 A씨가 한국 택시에서 겪은 일이 담긴... 
--------------------------------------------------
제목: "빛을 따라 걷는 순간"…심은진, 공항 게이트→여름 여정의 시작
본문: 팬들은 이번 사진에 대해 "공항 패션이 너무 세련됐다", "새로운 일상 여행이 기대된다", "여행길에서마저 카리스마가 묻어난다"라는 반응을 보이며 아낌없는 응원을 보내고 있다. 여름의 뜨거운 햇살만큼이나 짙

## 7. 판다스를 활용한 데이터 분석 (선택 사항)

저장된 데이터를 판다스 데이터프레임으로 변환하여 추가 분석을 수행할 수 있습니다.

In [ ]:
# 판다스 데이터프레임으로 변환
if response.status_code == 200 and 'items' in news_data:
    # 데이터 준비
    titles = []
    contents = []
    
    for item in news_data['items']:
        title = BeautifulSoup(item['title'].replace("<b>", "").replace("</b>", ""), "html.parser").text
        content = BeautifulSoup(item['description'].replace("<b>", "").replace("</b>", ""), "html.parser").text
        titles.append(title)
        contents.append(content)
    
    # 데이터프레임 생성
    df = pd.DataFrame({
        '제목': titles,
        '본문': contents
    })
    
    # 데이터프레임 출력
    display(df)
    
    # 데이터프레임을 CSV 파일로 저장 (선택 사항)
    # df.to_csv(f"{query}_news.csv", index=False, encoding='utf-8-sig')
    # print(f"'{query}_news.csv' CSV 파일로 저장되었습니다.")